# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cokezero20/FlyRank_AI_ML_Internship_NATIVIDAD/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Lane:** Binary classification — predict `is_declining_label` (1 = declining, 0 = not declining)

**Question shape:** Yes/no with an observed label → the skill says start with Logistic Regression, then Random Forest.

**Model 1 — Logistic Regression**
- Why first: readable coefficients, fast to train, establishes whether a linear boundary separates declining from non-declining content
- What it tells us: which features push toward decline and by how much

**Model 2 — Random Forest**
- Why second: captures non-linear interactions (e.g., low CTR only matters past a certain position threshold) that LR cannot
- What it tells us: whether added complexity earns a real improvement over the linear model

**Complexity rule:** Add the Random Forest only if it beats Logistic Regression. If LR already beats the baseline and RF adds little, LR wins — simplicity is a feature.

**Metric:** Precision@K (same as baseline) plus F1 and AUC for a fuller picture. Base rate reported alongside every metric.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Strategy:** Feature-label time separation.
- Features: January–March 2026 (what the model can see)
- Label: March vs April 2026 impressions (what the model predicts)

This prevents the model from seeing April data it's trying to predict.

**Split:** 80/20 random split at the content level (one row per content item). Random seed fixed for reproducibility.

**Why not time-based split:** The label already enforces a time boundary (features before April, label uses April). A second time split within Jan–Mar would shrink the training data without adding leakage protection.

In [2]:
import pandas as pd
import numpy as np
from datetime import date
from datasets import load_dataset
from google.colab import userdata
from sklearn.model_selection import train_test_split

SEED = 42
HF_TOKEN = userdata.get('HF_Token')

# ── Load data ────────────────────────────────────────────────────────
dataset = load_dataset(
    'FlyRank/internship-warehouse',
    name='fact_content_daily_performance',
    token=HF_TOKEN,
    streaming=True
)

print("Loading January–April 2026 data...")
data_rows = []
for batch in dataset['train'].iter(batch_size=100000):
    batch_df = pd.DataFrame(batch)
    if isinstance(batch_df['report_date'].iloc[0], str):
        batch_df['report_date'] = pd.to_datetime(batch_df['report_date']).dt.date
    data_batch = batch_df[
        (batch_df['report_date'] >= date(2026, 1, 1)) &
        (batch_df['report_date'] <= date(2026, 4, 30)) &
        (batch_df['ga4_data_available'] == True)
    ]
    if len(data_batch) > 0:
        data_rows.append(data_batch)

df = pd.concat(data_rows, ignore_index=True)
df['month'] = pd.to_datetime(df['report_date']).dt.month
print(f"Total rows: {len(df):,}")

# ── Create label (March vs April) ────────────────────────────────────
monthly_impr = (
    df[df['month'].isin([3, 4])]
    .groupby(['content_hash_id', 'month'])['gsc_impressions']
    .sum()
    .unstack(fill_value=0)
)
monthly_impr.columns = ['mar_impressions', 'apr_impressions']
monthly_impr['is_declining_label'] = (
    monthly_impr['apr_impressions'] < (0.8 * monthly_impr['mar_impressions'])
).astype(int)

print(f"Labeled content items: {len(monthly_impr):,}")
print(f"Decline rate: {monthly_impr['is_declining_label'].mean():.1%}")

# ── Build features from Jan–Mar only ─────────────────────────────────
df_features = df[df['month'].isin([1, 2, 3])]

features = ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position',
            'ga4_pageviews', 'ga4_sessions', 'ga4_users',
            'ga4_engaged_sessions', 'ga4_total_engagement_sec',
            'sessions_organic', 'sessions_direct', 'sessions_referral',
            'sessions_social', 'sessions_paid', 'sessions_ai', 'scroll_events']

content = df_features.groupby('content_hash_id')[features].agg(['sum', 'mean']).reset_index()
content.columns = ['content_hash_id'] + [f"{f}_{agg}" for f, agg in content.columns[1:]]

# Add CTR
content['ctr'] = content['gsc_clicks_sum'] / content['gsc_impressions_sum'].replace(0, np.nan)
content['ctr'] = content['ctr'].fillna(0)

# Merge label
content = content.merge(monthly_impr[['is_declining_label']], on='content_hash_id', how='inner')

# Drop rows with NaN
content = content.dropna()

feature_cols = [c for c in content.columns if c not in ['content_hash_id', 'is_declining_label']]
X = content[feature_cols]
y = content['is_declining_label']

print(f"\nFinal dataset: {len(X):,} content items, {len(feature_cols)} features")
print(f"Class split: {y.mean():.1%} declining, {1-y.mean():.1%} not declining")

# ── Train/test split ─────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f"\nTrain: {len(X_train):,}  |  Test: {len(X_test):,}")
print(f"Train decline rate: {y_train.mean():.3f}  |  Test decline rate: {y_test.mean():.3f}")


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading January–April 2026 data...
Total rows: 1,199,364
Labeled content items: 142,628
Decline rate: 24.9%

Final dataset: 66,926 content items, 31 features
Class split: 53.0% declining, 47.0% not declining

Train: 53,540  |  Test: 13,386
Train decline rate: 0.530  |  Test decline rate: 0.530


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, f1_score, roc_auc_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# ── Rebuild baseline on same test set ─────────────────────────────────

# Baseline rule: score = avg_position * visible (impressions >= 100)
test_content = content.loc[X_test.index].copy()
test_content['visible'] = (test_content['gsc_impressions_sum'] >= 100).astype(int)
test_content['baseline_score'] = test_content['gsc_avg_position_mean'] * test_content['visible']
test_content = test_content.sort_values('baseline_score', ascending=False)

def precision_at_k(y_true, y_scores, k):
    order = np.argsort(-np.asarray(y_scores))
    return np.asarray(y_true)[order[:k]].mean()

base_rate = y_test.mean()

print("=" * 60)
print("BASELINE (rule: avg_position * visible)")
print("=" * 60)
for k in [10, 20, 50, 100]:
    p = precision_at_k(
        test_content['is_declining_label'].values,
        test_content['baseline_score'].values, k
    )
    print(f"  Precision@{k}: {p:.3f}  (base rate: {base_rate:.3f})")

# ── Model 1: Logistic Regression ─────────────────────────────────────

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr = LogisticRegression(random_state=SEED, max_iter=1000)
lr.fit(X_train_scaled, y_train)

lr_probs = lr.predict_proba(X_test_scaled)[:, 1]
lr_preds = lr.predict(X_test_scaled)

print("\n" + "=" * 60)
print("MODEL 1: Logistic Regression")
print("=" * 60)
for k in [10, 20, 50, 100]:
    p = precision_at_k(y_test.values, lr_probs, k)
    print(f"  Precision@{k}: {p:.3f}  (base rate: {base_rate:.3f})")
print(f"  F1:  {f1_score(y_test, lr_preds):.3f}")
print(f"  AUC: {roc_auc_score(y_test, lr_probs):.3f}")

# ── Model 2: Random Forest ───────────────────────────────────────────

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=SEED,
    n_jobs=-1
)
rf.fit(X_train, y_train)

rf_probs = rf.predict_proba(X_test)[:, 1]
rf_preds = rf.predict(X_test)

print("\n" + "=" * 60)
print("MODEL 2: Random Forest")
print("=" * 60)
for k in [10, 20, 50, 100]:
    p = precision_at_k(y_test.values, rf_probs, k)
    print(f"  Precision@{k}: {p:.3f}  (base rate: {base_rate:.3f})")
print(f"  F1:  {f1_score(y_test, rf_preds):.3f}")
print(f"  AUC: {roc_auc_score(y_test, rf_probs):.3f}")

# ── Comparison Table ──────────────────────────────────────────────────

print("\n" + "=" * 60)
print("COMPARISON TABLE")
print("=" * 60)
print(f"{'Metric':<20} {'Base Rate':>10} {'Baseline':>10} {'Log Reg':>10} {'Rand Forest':>12}")
print("-" * 62)

for k in [10, 20, 50, 100]:
    b = precision_at_k(test_content['is_declining_label'].values,
                       test_content['baseline_score'].values, k)
    l = precision_at_k(y_test.values, lr_probs, k)
    r = precision_at_k(y_test.values, rf_probs, k)
    print(f"{'Precision@'+str(k):<20} {base_rate:>10.3f} {b:>10.3f} {l:>10.3f} {r:>12.3f}")

lr_f1 = f1_score(y_test, lr_preds)
rf_f1 = f1_score(y_test, rf_preds)
lr_auc = roc_auc_score(y_test, lr_probs)
rf_auc = roc_auc_score(y_test, rf_probs)

print(f"{'F1':<20} {'—':>10} {'—':>10} {lr_f1:>10.3f} {rf_f1:>12.3f}")
print(f"{'AUC':<20} {'—':>10} {'—':>10} {lr_auc:>10.3f} {rf_auc:>12.3f}")
print(f"\nBase rate: {base_rate:.3f}")
print(f"Random seed: {SEED}")


BASELINE (rule: avg_position * visible)
  Precision@10: 0.600  (base rate: 0.530)
  Precision@20: 0.600  (base rate: 0.530)
  Precision@50: 0.640  (base rate: 0.530)
  Precision@100: 0.700  (base rate: 0.530)

MODEL 1: Logistic Regression
  Precision@10: 0.700  (base rate: 0.530)
  Precision@20: 0.800  (base rate: 0.530)
  Precision@50: 0.880  (base rate: 0.530)
  Precision@100: 0.860  (base rate: 0.530)
  F1:  0.699
  AUC: 0.712

MODEL 2: Random Forest
  Precision@10: 0.800  (base rate: 0.530)
  Precision@20: 0.900  (base rate: 0.530)
  Precision@50: 0.920  (base rate: 0.530)
  Precision@100: 0.960  (base rate: 0.530)
  F1:  0.723
  AUC: 0.754

COMPARISON TABLE
Metric                Base Rate   Baseline    Log Reg  Rand Forest
--------------------------------------------------------------
Precision@10              0.530      0.600      0.700        0.800
Precision@20              0.530      0.600      0.800        0.900
Precision@50              0.530      0.640      0.880        0.92

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### Top 3 Features and Why They Make Sense

1. **gsc_impressions_mean** (0.0199) — Average daily impressions is the strongest signal. Content losing visibility day-to-day is the most direct indicator of decline. Makes sense: declining content shows up less.

2. **gsc_clicks_mean** (0.0128) — Average daily clicks. Content that stops earning clicks is losing relevance. Closely tied to impressions but captures the demand side — people aren't just not seeing it, they're not choosing it.

3. **gsc_avg_position_sum** (0.0065) — Accumulated position score. Worse position over time means the content is being pushed down in search results. This is what the baseline rule used, and it still matters — but the model learned it's the third signal, not the first.

### Error Patterns

**False positives (2,636 — predicted decline, actually stable):**
The 3 worst cases all share the same trait: low impressions (57–106), zero CTR, but the content didn't actually decline. The model sees "low activity + no clicks" and assumes decline, but these items were never performing — they're stable at near-zero. Same failure pattern as the baseline, just less frequent.

**False negatives (1,585 — missed real declines):**
The 3 worst misses are the opposite: high impressions (1,580–12,588), strong position (1.9–3.2), low but nonzero CTR. The model sees "high visibility + good position" and assumes healthy — but these items declined anyway. These are content items that looked fine in Jan–Mar but dropped in April for reasons not visible in the features (algorithm update, seasonal shift, competitor content).

### Summary

The model's blind spots mirror the baseline's but are sharper. It struggles at both extremes: content too small to read a trend from (false positives), and content that looks healthy but declines from external causes the features can't capture (false negatives). A trend feature — impression change over time rather than just average level — would likely fix the first problem. The second may require external signals the dataset doesn't contain.

### Model Choice

Random Forest earns its complexity. It beats Logistic Regression at every metric (AUC 0.754 vs 0.712, F1 0.723 vs 0.699) and substantially beats the baseline at precision@K. The improvement is consistent, not concentrated at one K value — this is a real gain, not noise.

In [4]:
# ── Feature Importance (Permutation) ──────────────────────────────────
from sklearn.inspection import permutation_importance

perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=SEED, n_jobs=-1)

imp_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': perm.importances_mean
}).sort_values('importance', ascending=False)

print("TOP 10 FEATURES (permutation importance)")
print("=" * 50)
for i, row in imp_df.head(10).iterrows():
    print(f"  {row['feature']:<40} {row['importance']:.4f}")

# ── Error Analysis ────────────────────────────────────────────────────

test_results = content.loc[X_test.index][['content_hash_id']].copy()
test_results['y_true'] = y_test.values
test_results['y_pred'] = rf_preds
test_results['y_prob'] = rf_probs

# Where is the model most wrong?
# False positives: predicted declining but actually not
fp = test_results[(test_results['y_pred'] == 1) & (test_results['y_true'] == 0)]
# False negatives: predicted not declining but actually declining
fn = test_results[(test_results['y_pred'] == 0) & (test_results['y_true'] == 1)]

print(f"\nERROR BREAKDOWN")
print("=" * 50)
print(f"  False positives (predicted decline, actually stable): {len(fp)}")
print(f"  False negatives (missed real declines):              {len(fn)}")
print(f"  Total errors: {len(fp) + len(fn)} out of {len(test_results)}")

# ── 3 Concrete Wrong Cases ────────────────────────────────────────────

# Merge features back for inspection
errors = test_results[test_results['y_pred'] != test_results['y_true']].copy()
errors = errors.merge(content[['content_hash_id'] + feature_cols], on='content_hash_id')

# 3 worst false positives (highest confidence, still wrong)
print("\n3 WRONG CASES — False Positives (confident but wrong)")
print("=" * 50)
worst_fp = errors[errors['y_true'] == 0].nlargest(3, 'y_prob')
for _, row in worst_fp.iterrows():
    print(f"\n  {row['content_hash_id']}")
    print(f"    Predicted prob: {row['y_prob']:.3f}  |  Actual: not declining")
    print(f"    Impressions (sum): {row['gsc_impressions_sum']:.0f}")
    print(f"    Avg position: {row['gsc_avg_position_mean']:.1f}")
    print(f"    CTR: {row['ctr']:.4f}")

# 3 worst false negatives (lowest probability, but actually declining)
print("\n3 WRONG CASES — False Negatives (missed declines)")
print("=" * 50)
worst_fn = errors[errors['y_true'] == 1].nsmallest(3, 'y_prob')
for _, row in worst_fn.iterrows():
    print(f"\n  {row['content_hash_id']}")
    print(f"    Predicted prob: {row['y_prob']:.3f}  |  Actual: declining")
    print(f"    Impressions (sum): {row['gsc_impressions_sum']:.0f}")
    print(f"    Avg position: {row['gsc_avg_position_mean']:.1f}")
    print(f"    CTR: {row['ctr']:.4f}")


TOP 10 FEATURES (permutation importance)
  gsc_impressions_mean                     0.0199
  gsc_clicks_mean                          0.0128
  gsc_avg_position_sum                     0.0065
  sessions_direct_sum                      0.0054
  gsc_impressions_sum                      0.0047
  sessions_direct_mean                     0.0042
  sessions_paid_mean                       0.0040
  ctr                                      0.0036
  ga4_total_engagement_sec_mean            0.0026
  sessions_organic_mean                    0.0026

ERROR BREAKDOWN
  False positives (predicted decline, actually stable): 2636
  False negatives (missed real declines):              1585
  Total errors: 4221 out of 13386

3 WRONG CASES — False Positives (confident but wrong)

  content_755d44e92cdc65c0
    Predicted prob: 0.943  |  Actual: not declining
    Impressions (sum): 104
    Avg position: 6.7
    CTR: 0.0000

  content_72ed4efb5219b4c0
    Predicted prob: 0.943  |  Actual: not declining
    Imp

In [5]:
import sklearn
print(f"scikit-learn: {sklearn.__version__}")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")

scikit-learn: 1.6.1
pandas: 2.2.3
numpy: 2.1.3


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.